In [1]:
!pip install -q gradio transformers torch reportlab

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 38.7 MB/s eta 0:00:00


In [2]:
schema = {
    "users": ["id", "name", "email", "signup_date", "age", "country", "status"],
    "orders": ["id", "user_id", "product_name", "amount", "order_date", "status"],
    "products": ["id", "name", "price", "category", "stock"],
    "transactions": ["id", "user_id", "amount", "date", "type"]
}

In [3]:
def is_safe(user_input):
    dangerous_keywords = ["DROP", "DELETE", "ALTER", "INSERT", "UPDATE", "EXEC"]
    dangerous_patterns = [";", "--", "/*", "*/"]

    for keyword in dangerous_keywords:
        if keyword.lower() in user_input.lower():
            return False

    for pattern in dangerous_patterns:
        if pattern in user_input:
            return False

    return True

In [4]:
print(is_safe("Show users"))
print(is_safe("DROP TABLE users;"))

True
False


In [5]:
def get_table_name(user_input):
    user_input = user_input.lower()

    if any(word in user_input for word in ["user", "customer", "account"]):
        return "users"

    elif any(word in user_input for word in ["order", "purchase"]):
        return "orders"

    elif any(word in user_input for word in ["product", "item"]):
        return "products"

    elif any(word in user_input for word in ["transaction", "payment"]):
        return "transactions"

    else:
        return None

In [6]:
print(get_table_name("Show users"))
print(get_table_name("Find transactions"))

users
transactions


In [7]:
import re
from datetime import datetime

def extract_conditions(user_input, table):
    conditions = []

    user_input = user_input.lower()

    # Amount condition
    amount_match = re.search(r'greater than (\d+)', user_input)
    if amount_match:
        amount = amount_match.group(1)
        if table in ["transactions", "orders"]:
            conditions.append(f"amount > {amount}")
        elif table == "products":
            conditions.append(f"price > {amount}")

    # Last week condition
    if "last week" in user_input:
        if table == "transactions":
            conditions.append("DATE(date) >= DATE('now', '-7 days')")
        elif table == "users":
            conditions.append("DATE(signup_date) >= DATE('now', '-7 days')")
        elif table == "orders":
            conditions.append("DATE(order_date) >= DATE('now', '-7 days')")

    # Last month condition
    if "last month" in user_input:
        if table == "users":
            conditions.append("DATE(signup_date) >= DATE('now', '-1 month')")
        elif table == "transactions":
            conditions.append("DATE(date) >= DATE('now', '-1 month')")

    return conditions

In [8]:
def generate_sql(user_input):

    if not is_safe(user_input):
        return "Error: Unsafe query detected."

    table = get_table_name(user_input)

    if not table:
        return "Error: Could not detect table."

    conditions = extract_conditions(user_input, table)

    query = f"SELECT * FROM {table}"

    if conditions:
        query += " WHERE " + " AND ".join(conditions)

    query += " LIMIT 10;"

    return query

In [9]:
print(generate_sql("Show users who signed up last month"))
print(generate_sql("Find transactions greater than 1000 from last week"))
print(generate_sql("DROP TABLE users;"))

SELECT * FROM users WHERE DATE(signup_date) >= DATE('now', '-1 month') LIMIT 10;
SELECT * FROM transactions WHERE amount > 1000 AND DATE(date) >= DATE('now', '-7 days') LIMIT 10;
Error: Unsafe query detected.


In [10]:
import gradio as gr

def chat_interface(user_input):
    return generate_sql(user_input)

demo = gr.Interface(
    fn=chat_interface,
    inputs=gr.Textbox(placeholder="Enter your request in plain English..."),
    outputs="text",
    title="SQL Query Generator",
    description="Natural Language to SQL Query Generator"
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9ec935c16dba5c24e6.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
